<a href="https://colab.research.google.com/github/nitinog10/Mini-SLM/blob/main/Minislm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install all required libraries
!pip install torch transformers datasets tokenizers numpy tqdm

In [2]:
# 2. Download and prepare the wikitext-2-raw-v1 dataset
from datasets import load_dataset

dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
train_texts = [x['text'] for x in dataset['train'] if len(x['text']) > 0]
val_texts = [x['text'] for x in dataset['validation'] if len(x['text']) > 0]

print(f"Loaded {len(train_texts)} training samples.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Loaded 23767 training samples.


In [3]:
# 3. Train a BPE tokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(vocab_size=8000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
tokenizer.pre_tokenizer = Whitespace()

# Train the tokenizer on the training text
tokenizer.train_from_iterator(train_texts, trainer)
tokenizer.save("tokenizer.json")
print("Tokenizer trained and saved.")

Tokenizer trained and saved.


In [4]:
# 4. Build a GPT-style Transformer model in PyTorch
import torch
import torch.nn as nn
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SLMConfig:
    vocab_size = 8000
    emb_dim = 256
    n_layers = 4
    n_heads = 4
    ff_dim = 1024
    max_len = 256
    dropout = 0.1

class MultiHeadAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.emb_dim // cfg.n_heads
        self.wq = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wk = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wv = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.wo = nn.Linear(cfg.emb_dim, cfg.emb_dim)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, mask=None):
        b, t, c = x.size()
        q = self.wq(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(b, t, c)
        return self.wo(out)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.emb_dim)
        self.attn = MultiHeadAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.emb_dim)
        self.ff = nn.Sequential(
            nn.Linear(cfg.emb_dim, cfg.ff_dim),
            nn.GELU(),
            nn.Linear(cfg.ff_dim, cfg.emb_dim),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.max_len, cfg.emb_dim))
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.emb_dim)
        self.head = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        b, t = idx.size()
        mask = torch.tril(torch.ones(t, t)).to(device)
        x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        for block in self.blocks:
            x = block(x, mask)
        logits = self.head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

cfg = SLMConfig()
model = SLM(cfg).to(device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 7,321,088


In [5]:
# 5. Efficient DataLoader
import numpy as np

def get_batches(data, batch_size, seq_len):
    # Flatten data and chunk it
    n_batches = len(data) // (batch_size * seq_len)
    data = data[:n_batches * batch_size * seq_len]
    data = data.reshape(batch_size, -1)
    for i in range(0, data.shape[1] - seq_len, seq_len):
        x = data[:, i:i+seq_len]
        y = data[:, i+1:i+seq_len+1]
        yield torch.tensor(x, dtype=torch.long).to(device), torch.tensor(y, dtype=torch.long).to(device)

# Tokenize full dataset once
all_tokens = []
for text in train_texts:
    all_tokens.extend(tokenizer.encode(text).ids)
train_data = np.array(all_tokens)

In [6]:
# 6. Full Training Loop
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=3e-4)
model.train()

epochs = 3
step = 0
for epoch in range(epochs):
    batch_gen = get_batches(train_data, batch_size=32, seq_len=cfg.max_len)
    for x, y in batch_gen:
        logits, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")
        step += 1

Epoch 1 | Step 0 | Loss: 9.1509
Epoch 1 | Step 100 | Loss: 6.6833
Epoch 1 | Step 200 | Loss: 6.3459
Epoch 1 | Step 300 | Loss: 6.1894
Epoch 2 | Step 400 | Loss: 5.9035
Epoch 2 | Step 500 | Loss: 5.7389
Epoch 2 | Step 600 | Loss: 5.6575
Epoch 3 | Step 700 | Loss: 5.5540
Epoch 3 | Step 800 | Loss: 5.5254
Epoch 3 | Step 900 | Loss: 5.3988


In [7]:
# 7. Text Generation Function
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, k=10):
    model.eval()
    ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):
        # Crop context if needed
        idx_cond = ids[:, -cfg.max_len:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] # Get last token logits

        # Top-k sampling
        v, _ = torch.topk(logits, k)
        logits[logits < v[:, [-1]]] = -float('Inf')
        probs = torch.softmax(logits, dim=-1)

        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat((ids, next_id), dim=1)

    return tokenizer.decode(ids[0].tolist())

In [8]:
# 8. Test the model
prompt = "The history of science"
generated_text = generate(model, tokenizer, prompt)
print(f"Prompt: {prompt}\nGenerated: {generated_text}")

Prompt: The history of science
Generated: The history of science fiction ( a , and is a large , and the same year . The first , the main characters and has not been described as " and a large @-@ year , and that this was not a large species of the species of the species , with the other in a " . " , and the " . A g le te , in the b ble . E . " . The g le a " is " , the species are not only , in the species of a " , and are " a


In [9]:
# 9. Save and Download model
torch.save(model.state_dict(), 'slm_model.pt')
from google.colab import files
# files.download('slm_model.pt') # Uncomment to trigger browser download

In [11]:
# 10. Generate text with a different prompt
new_prompt = "In the distant future, artificial intelligence"
new_generated_text = generate(model, tokenizer, new_prompt, max_new_tokens=100)
print(f"Prompt: {new_prompt}\nGenerated: {new_generated_text}")

Prompt: In the distant future, artificial intelligence
Generated: In the dist ant future , art ificial intelligence as the first to take a large @-@ up of the second , and the city of the Jin , in the Chinese to the Chinese . As of the Chinese . In the brigade , the Song was the brigade , the Chinese in the first of the Jin . On the Song of the brigade was in a tropical storm . The ship . The ship ' s the first of the Jin and the Jin and then to the Song . The next day of the 130 th Engineer Brigade was not ably the first of the


In [13]:
new_prompt = "what is artificial intelligence"
new_generated_text = generate(model, tokenizer, new_prompt, max_new_tokens=100)
print(f"Prompt: {new_prompt}\nGenerated: {new_generated_text}")

Prompt: what is artificial intelligence
Generated: what is art ificial intelligence to have been in the novel in the book . The most common and the first book of the first time , which is also used for the first @-@ old , in the novel of the novel , which was also known as the book of its original " and that " . = = = The Good Ter ror ist , was the American , the first book ' t issue of the " . He has been released on the film ' s book was a series ' s a " the series of a series to
